# DengueCast LK — run the Streamlit dashboard in Google Colab

This notebook is self-contained. It uploads the project, retrains the models with Colab's library versions (the
`.pkl` files only load with the same scikit-learn version they were saved with), then starts the dashboard.
Run the cells from top to bottom.

### Step 1 — go to the project folder and install Streamlit

In [ ]:
import os
if not os.path.exists("/content/denguecast/app.py"):
    from google.colab import files
    print("Project folder not found. Upload DengueCast_LK_submission.zip ...")
    up = files.upload(); z = next(iter(up))
    !unzip -q -o "{z}" -d /content
%cd /content/denguecast
!pip install -q streamlit plotly xgboost lightgbm catboost
print("models:", sorted(os.listdir("models")))

### Step 1b — retrain the models with Colab's versions (about 4–5 minutes)
This runs the main notebook in the background and rewrites `models/*.pkl` and `results/`.

In [ ]:
!jupyter nbconvert --to notebook --execute DengueCast_LK_Ensemble_Learning.ipynb \
      --output DengueCast_LK_executed_in_colab.ipynb --ExecutePreprocessor.timeout=1800
import pandas as pd
pd.read_csv("results/model_comparison.csv")[["model", "MAE", "RMSE", "R2"]].round(3)

### Step 2 — start the dashboard in the background

In [ ]:
!pkill -f "streamlit run" || true
!nohup streamlit run app.py --server.port 8501 --server.headless true \
      --server.enableCORS false --server.enableXsrfProtection false > streamlit.log 2>&1 &
import time; time.sleep(8)
!tail -5 streamlit.log

### Step 3 (option A) — open it through Colab's own proxy
Click the link that appears. If the page stays blank or keeps loading, use option B.

In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(8501)

### Step 3 (option B) — public link with localtunnel
1. Run the cell. It prints a **password** (your Colab IP) and a URL like `https://xxxx.loca.lt`.
2. Open the URL, paste the password into the "Tunnel Password" box, click **Submit**.
3. Stop the cell (■) when you are done.

In [ ]:
import urllib.request
print("Tunnel password:", urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip())
!npx --yes localtunnel --port 8501

### Step 4 — if something goes wrong
Show the last lines of the Streamlit log:

In [ ]:
!tail -30 streamlit.log

### Step 5 — download the project for GitHub / Streamlit Cloud
The `.pkl` files made in Colab only load with Colab's library versions. This cell writes a `requirements.txt` with
those exact versions and downloads the whole project as a zip, ready to push to GitHub.

In [ ]:
from importlib.metadata import version
pkgs = ["pandas", "numpy", "scikit-learn", "scipy", "xgboost", "lightgbm", "catboost", "statsmodels",
        "joblib", "streamlit", "plotly", "matplotlib"]
open("requirements.txt", "w").write("\n".join(f"{p}=={version(p)}" for p in pkgs) + "\n")
print(open("requirements.txt").read())
!cd /content && rm -f denguecast_colab.zip && zip -q -r denguecast_colab.zip denguecast -x "*/__pycache__/*" "*/catboost_info/*" "*/streamlit.log"
from google.colab import files
files.download("/content/denguecast_colab.zip")